# Environment Verification

This notebook verifies that the python environment is set up correctly, has access to the GPU/CUDA via PyTorch, and works with `ipykernel`.

In [ ]:
import torch
import torch.nn as nn

# grid extraction or unfolding

class PatchEmbedding(nn.Module):
    def __init__(self, height, width, in_channels=3, patch_size=16, embed_dim=768):
        super(PatchEmbedding, self).__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.num_patches = (height // patch_size) * (width // patch_size)
        
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size
        )
        # learnable class token
        self.cls_token = nn.Parameter(torch.zeros(1, embed_dim))
        # learnable position embeddings
        self.embeddings = nn.Parameter(torch.zeros(self.num_patches + 1, embed_dim))


    def _init_weights(self):
        nn.init.xavier_uniform_(self.proj.weight)
        if self.proj.bias is not None:
            nn.init.zeros_(self.proj.bias)
        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.embeddings, std=0.02)

    def forward(self, x):
        C, H, W = x.shape

        assert C == self.proj.in_channels, f"Expected input channels {self.proj.in_channels}, but got {C}."
        assert H % self.patch_size == 0, f"Height {H} must be divisible by patch size {self.patch_size}."
        assert W % self.patch_size == 0, f"Width {W} must be divisible by patch size {self.patch_size}."

        # Extract patches
        x = self.proj(x)      # (C, H, W) -> (D, H//P, W//P)
        x = x.flatten(1)      # (D, H//P, W//P) ->(D, NumPatches)
        x = x.transpose(0, 1) # (D, NumPatches) -> (NumPatches, D)

        # Prepend with class token
        x = torch.cat((self.cls_token, x), dim=0)  # (1 + NumPatches, D)

        # Add position embeddings
        x = x + self.embeddings # (1 + NumPatches, D) + (1 + NumPatches, D) -> (1 + NumPatches, D)

        return x
    
class MultiheadSelfAttention(nn.Module):
    def __init__(self, input_dim, num_heads, head_dim):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim

        self.qkv = nn.Linear(input_dim, num_heads * head_dim * 3, bias=False)
        
        self.o = nn.Linear(self.num_heads * self.head_dim, input_dim)

        nn.init.xavier_uniform_(self.qkv.weight)
        nn.init.xavier_uniform_(self.o.weight)

    def forward(self, x):
        # Sl = sequence length
        # D = input_dim
        # Hn = num_heads
        # Hd = head_dim
        qkv = self.qkv(x) # (Sl, D) -> (Sl, Hn * Hd * 3)

        q, k, v = qkv.chunk(3, dim=-1) # (Sl, Hn * Hd)
        q = q.view(q.shape[0], self.num_heads, self.head_dim) # (Sl, Hn, Hd)
        k = k.view(k.shape[0], self.num_heads, self.head_dim) # (Sl, Hn, Hd)
        v = v.view(v.shape[0], self.num_heads, self.head_dim) # (Sl, Hn, Hd)
        
        q = q.permute(1, 0, 2) # (Hn, Sl, Hd)
        k = k.permute(1, 0, 2) # (Hn, Sl, Hd)
        v = v.permute(1, 0, 2) # (Hn, Sl, Hd)
        logits = q @ k.transpose(-2, -1) # (Hn, Sl, Hd) @ (Hn, Hd, Sl) -> (Hn, Sl, Sl)
        A = torch.softmax(logits / (self.head_dim ** 0.5), dim=-1) # (Hn, Sl, Sl)
        sa = A @ v # (Hn, Sl, Sl) @ (Hn, Sl, Hd) -> (Hn, Sl, Hd)
        
        sa = sa.permute(1, 0, 2).contiguous() # (Sl, Hn, Hd)
        sa = sa.view(sa.shape[0], self.num_heads * self.head_dim) # (Sl, Hn, Hd) -> (Sl, Hn * Hd)

        return self.o(sa) # (Sl, Hn * Hd) -> (Sl, D)

class MLP(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=4.0):
        super().__init__()
        hidden_dim = int(embed_dim * mlp_ratio)

        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, embed_dim)

        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)

    def forward(self, x):
        # x: (Sl, D)
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)

        return x
    
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, head_dim, mlp_ratio=4.0):
        super().__init__()
        # Sub-layer 1: Multi-head Self-Attention
        self.norm1 = nn.LayerNorm(embed_dim)
        self.msa = MultiheadSelfAttention(embed_dim, num_heads, head_dim)
        # Sub-layer 2: MLP
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_ratio)

    def forward(self, x):
        # Attention with Pre-Ln and Residual Connection
        x_norm = self.norm1(x)
        attn_out = self.msa(x_norm)
        x = x + attn_out

        # MLP with Pre-Ln and Residual Connection
        x_norm = self.norm2(x)
        mlp_out = self.mlp(x_norm)
        x = x + mlp_out

        return x
    
class VisionTransformer(nn.Module):
    def __init__(self, image_size=224, patch_size=16, in_channels=3, num_classes=10, embed_dim=768, depth=12, num_heads=12, head_dim=64, mlp_ratio=4.0):
        super().__init__()
        self.patch_embed = PatchEmbedding(image_size, image_size, in_channels, patch_size, embed_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, head_dim, mlp_ratio) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.zeros_(self.head.bias)
        nn.init.zeros_(self.head.weight)

    def forward(self, x):
        x = self.patch_embed(x) # (1 + NumPatches, D)
        for block in self.blocks:
            x = block(x) # (1 + NumPatches, D)
        x = self.norm(x) # (1 + NumPatches, D)
        cls_token_final = x[0] # (D,)
        logits = self.head(cls_token_final) # (D,) -> (num_classes,)
        return logits